# 03 · Pandas: จัดการตารางข้อมูลสถานีวัดน้ำฝน
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apichayaice7835-png/Teach-Python-Geography/blob/main/03_Pandas_Rainfall_Stations.ipynb)

**เวลา:** 3 ชั่วโมง | **ระดับ:** 🟡 ปานกลาง (มีส่วน 🔴 ยาก)

### ข้อมูลที่ใช้ (โฟลเดอร์ `data/`)
| ไฟล์ | เนื้อหา | คอลัมน์ |
|---|---|---|
| `rain_stations_north.csv` | ตำแหน่งสถานี 16 แห่ง ภาคเหนือ | station_id, name_th, name_en, province, lat, lon, elev_m |
| `rain_monthly_2020_2024.csv` | ฝน/อุณหภูมิรายเดือน 5 ปี | station_id, date, rain_mm, tmean_c |

> ⚠️ **ข้อมูลเป็นข้อมูลจำลองเพื่อการสอน** (พิกัดใกล้เคียงอำเภอจริง แต่ค่าฝนสร้างจากแบบจำลอง) มี *ค่าว่าง (missing)* แทรกไว้โดยตั้งใจเพื่อฝึกทำความสะอาดข้อมูล
> เมื่อได้ข้อมูลจริงจากกรมอุตุนิยมวิทยาหรือกรมชลประทาน ให้จัดคอลัมน์ให้ตรงกัน แล้วใช้โค้ดเดิมได้ทันที

**Workflow:** `อ่านไฟล์ → สำรวจ → ทำความสะอาด → รวมตาราง (merge) → สรุป (groupby) → ส่งออก`

In [ ]:
# ============================================================
# เซลล์โหลดข้อมูล (ใช้ซ้ำทุก Notebook)
# ลำดับการหา: 1) โฟลเดอร์ data/ ในเครื่อง  2) GitHub ของอาจารย์
# ============================================================
import os
import pandas as pd

GITHUB_USER = "apichayaice7835-png"      # ← ชื่อบัญชี GitHub ของอาจารย์
REPO        = "Teach-Python-Geography"   # ← ชื่อ repository
BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data/"

def data_path(filename):
    """คืน path ของไฟล์: ถ้ามีในเครื่องใช้ในเครื่อง ถ้าไม่มีใช้ลิงก์ GitHub"""
    for folder in ["data", f"{REPO}/data", "/content/data"]:
        p = os.path.join(folder, filename)
        if os.path.exists(p):
            return p
    return BASE_URL + filename

print("ไฟล์สถานี :", data_path("rain_stations_north.csv"))
print("ไฟล์ฝนรายเดือน:", data_path("rain_monthly_2020_2024.csv"))

## 🟢 1. อ่านและสำรวจข้อมูล

In [ ]:
stations = pd.read_csv(data_path("rain_stations_north.csv"))
rain     = pd.read_csv(data_path("rain_monthly_2020_2024.csv"))

print("stations:", stations.shape, "| rain:", rain.shape)   # (จำนวนแถว, จำนวนคอลัมน์)
stations.head()

In [ ]:
rain.info()        # ชนิดข้อมูลแต่ละคอลัมน์ และจำนวนค่าที่ไม่ว่าง

In [ ]:
rain.describe()    # สถิติพื้นฐาน: count, mean, std, min, quartiles, max

## 🟢 2. เลือกคอลัมน์และกรองแถว

In [ ]:
# เลือกคอลัมน์
print(stations[["name_th", "elev_m"]].head(3))

# กรองแถวตามเงื่อนไข (เหมือน Select by Attribute)
high = stations[stations["elev_m"] > 350]
print("\nสถานีที่สูงกว่า 350 ม.:\n", high[["name_th", "province", "elev_m"]])

nan_prov = stations[stations["province"] == "Nan"]
print("\nสถานีในจังหวัดน่าน:", nan_prov["name_th"].tolist())

## 🟡 3. จัดการวันที่ (datetime)

In [ ]:
rain["date"]  = pd.to_datetime(rain["date"])     # ข้อความ → วันที่
rain["year"]  = rain["date"].dt.year
rain["month"] = rain["date"].dt.month
rain.head()

## 🟡 4. ทำความสะอาดข้อมูล: ค่าว่าง (Missing values)

In [ ]:
print("จำนวนค่าว่างแต่ละคอลัมน์:\n", rain.isna().sum())
print("\nตัวอย่างแถวที่ว่าง:")
rain[rain["rain_mm"].isna()].head()

In [ ]:
# วิธีเติมค่าที่เหมาะกับข้อมูลภูมิอากาศ: ใช้ "ค่าเฉลี่ยของสถานีเดียวกัน เดือนเดียวกัน" (climatological mean)
# transform() คืนค่าที่มีขนาดเท่ากับตารางเดิม จึงเติมกลับได้ทันที
clim = rain.groupby(["station_id", "month"])["rain_mm"].transform("mean")
rain["filled"]  = rain["rain_mm"].isna()          # จดไว้ว่าแถวไหนถูกเติม (โปร่งใสทางวิชาการ)
rain["rain_mm"] = rain["rain_mm"].fillna(clim.round(1))
print("ค่าว่างที่เหลือ:", rain["rain_mm"].isna().sum(), "| จำนวนที่เติม:", rain["filled"].sum())

## 🟡 5. รวมตาราง (merge) = Join ใน GIS
เชื่อมตารางฝนกับตารางสถานีด้วยคอลัมน์ร่วม `station_id`

In [ ]:
df = rain.merge(stations, on="station_id", how="left")
df.head()

## 🟡 6. สรุปข้อมูลด้วย `groupby`

In [ ]:
# 6.1 ฝนรวมรายปีของแต่ละสถานี
annual = df.groupby(["station_id", "name_th", "year"])["rain_mm"].sum().reset_index()
annual.head()

In [ ]:
# 6.2 ค่าเฉลี่ยฝนรายปี 5 ปี เรียงจากมากไปน้อย
mean_annual = (annual.groupby(["station_id", "name_th"])["rain_mm"]
                     .mean().round(0)
                     .sort_values(ascending=False)
                     .reset_index(name="mean_annual_mm"))
mean_annual

In [ ]:
# 6.3 ฝนรายปีเฉลี่ยของทุกสถานี: ปีไหนฝนมาก/น้อย
annual.groupby("year")["rain_mm"].mean().round(0)

## 🟡 7. Pivot table: ค่าเฉลี่ยรายเดือน (Climatology) แถว = สถานี, คอลัมน์ = เดือน

In [ ]:
climatology = df.pivot_table(index="name_th", columns="month", values="rain_mm", aggfunc="mean").round(0)
climatology

## 🔴 8. จำแนกฤดูกาล และคำนวณ Anomaly
ฤดูกาลแบบประมาณตามเดือน: ฤดูหนาว (พ.ย.–ก.พ.), ฤดูร้อน (มี.ค.–เม.ย.), ฤดูฝน (พ.ค.–ต.ค.)

In [ ]:
season_map = {11:"หนาว", 12:"หนาว", 1:"หนาว", 2:"หนาว",
              3:"ร้อน", 4:"ร้อน",
              5:"ฝน", 6:"ฝน", 7:"ฝน", 8:"ฝน", 9:"ฝน", 10:"ฝน"}
df["season"] = df["month"].map(season_map)

season_share = df.pivot_table(index="name_th", columns="season", values="rain_mm", aggfunc="sum")
season_share = (season_share.div(season_share.sum(axis=1), axis=0) * 100).round(1)   # เป็นร้อยละ
season_share.sort_values("ฝน", ascending=False)

In [ ]:
# Anomaly รายปี (%) = (ฝนปีนั้น − ค่าเฉลี่ย 5 ปี) / ค่าเฉลี่ย × 100
annual["mean5"] = annual.groupby("station_id")["rain_mm"].transform("mean")
annual["anomaly_pct"] = ((annual["rain_mm"] - annual["mean5"]) / annual["mean5"] * 100).round(1)
anom_table = annual.pivot(index="name_th", columns="year", values="anomaly_pct")
anom_table

In [ ]:
# 🔴 ความสัมพันธ์ระหว่างความสูงกับฝนรายปี (Pearson correlation)
merged = mean_annual.merge(stations[["station_id", "elev_m", "lon"]], on="station_id")
print("r(ความสูง, ฝน)     = %.2f" % merged["mean_annual_mm"].corr(merged["elev_m"]))
print("r(ลองจิจูด, ฝน)    = %.2f" % merged["mean_annual_mm"].corr(merged["lon"]))
print("⚠️ Correlation ≠ Causation และตัวอย่างมีเพียง 16 สถานี")

## 9. ส่งออกผลลัพธ์

In [ ]:
os.makedirs("output", exist_ok=True)
mean_annual.merge(stations, on=["station_id", "name_th"]).to_csv(
    "output/station_mean_annual_rain.csv", index=False, encoding="utf-8-sig")   # utf-8-sig เปิดใน Excel แล้วภาษาไทยไม่เพี้ยน
print("บันทึกแล้ว → output/station_mean_annual_rain.csv")

## 📎 เมื่อใช้ไฟล์จริงของตนเอง (เคล็ดลับ)
```python
pd.read_csv("file.csv", encoding="cp874")        # ไฟล์ภาษาไทยจาก Windows/Excel รุ่นเก่า
pd.read_excel("file.xlsx", sheet_name="2567")    # ไฟล์ Excel
df = df.rename(columns={"รหัสสถานี":"station_id", "ละติจูด":"lat", "ลองจิจูด":"lon"})
df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")   # ค่าอย่าง "T" (trace), "-" → NaN
```
> ข้อมูลฝนรายวันจากหน่วยงานมักใช้ `T` แทน *ฝนเล็กน้อยวัดไม่ได้* (< 0.1 มม.) ควรตัดสินใจให้ชัดว่าจะแทนด้วย 0 หรือ NaN

---
## ✏️ แบบฝึกหัด
**🟢** 1) หาสถานีที่อยู่ต่ำที่สุด  2) กรองข้อมูลเฉพาะปี 2022 ของสถานีน่าน (S13)

**🟡** 3) หาเดือนที่ฝนมากที่สุดของแต่ละสถานี (hint: `idxmax`)
4) คำนวณอุณหภูมิเฉลี่ยรายปีของแต่ละจังหวัด (`groupby("province")`)

**🔴** 5) นับจำนวน "เดือนแล้ง" (ฝน < 50 มม.) ต่อปีของแต่ละสถานี แล้วหาว่าปีไหนแล้งที่สุด
6) หาสถานีที่มีค่าว่างมากที่สุด และอภิปรายว่าการเติมค่าด้วยค่าเฉลี่ยมีข้อจำกัดอะไร